## Previsão de Demanda

Tarefa:
1. Utilize o dataset vendas_2023_2024.csv
2. Construa um modelo baseline simples, utilizando: Média móvel dos últimos 7 dias de vendas (considerando apenas dados anteriores à data prevista).
3. Gere a previsão diária de vendas para Janeiro de 2024.
4. Compare as previsões com os valores reais do período de teste utilizando a métrica: MAE — Mean Absolute Error
5. Responda objetivamente:
     - a. O baseline é adequado para esse produto?
     - b. Cite uma limitação desse método.

In [1]:
import pandas as pd
import numpy as np

df_vendas = pd.read_csv('../datasets/vendas_2023_2024.csv')

# Parse de datas
df_vendas['sale_date'] = pd.to_datetime(df_vendas['sale_date'], dayfirst=True, errors='coerce')
df_vendas = df_vendas.dropna(subset=['sale_date'])

# Produto alvo
PRODUTO = 'Motor de Popa Yamaha Evo Dash 155HP'

# Join com produtos para filtrar pelo nome
df_produtos = pd.read_csv('../datasets/produtos_raw.csv').rename(columns={'code': 'id_product'})
id_produto = df_produtos[df_produtos['name'] == PRODUTO]['id_product'].values[0]

df_produto = df_vendas[df_vendas['id_product'] == id_produto].copy()
print(f"Produto: {PRODUTO} | id_product: {id_produto}")
print(f"Total de transações: {len(df_produto)}")

Produto: Motor de Popa Yamaha Evo Dash 155HP | id_product: 54
Total de transações: 8


In [2]:
# Agrega vendas por dia
vendas_diarias = (
    df_produto.groupby('sale_date')['qtd']
    .sum()
    .reset_index()
    .rename(columns={'qtd': 'vendas'})
)

# Calendário completo do período
calendario = pd.DataFrame({
    'sale_date': pd.date_range(
        df_vendas['sale_date'].min(),
        df_vendas['sale_date'].max()
    )
})

serie = calendario.merge(vendas_diarias, on='sale_date', how='left').fillna(0)
serie['vendas'] = serie['vendas'].astype(float)

print(f"Período: {serie['sale_date'].min().date()} → {serie['sale_date'].max().date()}")
print(f"Total de dias: {len(serie)}")

Período: 2023-01-01 → 2024-12-12
Total de dias: 712


In [3]:
treino = serie[serie['sale_date'] <= '2023-12-31'].copy()
teste  = serie[(serie['sale_date'] >= '2024-01-01') &
               (serie['sale_date'] <= '2024-01-31')].copy()

print(f"Treino: {len(treino)} dias | Teste: {len(teste)} dias")

Treino: 365 dias | Teste: 31 dias


In [4]:
# Concatena treino + teste para calcular a janela sem data leakage
serie_completa = pd.concat([treino, teste]).reset_index(drop=True)
serie_completa['previsao'] = (
    serie_completa['vendas']
    .shift(1)                        # shift garante que o dia atual não entra na janela
    .rolling(window=7, min_periods=1)
    .mean()
    .round(2)
)

# Extrai apenas o período de teste com a previsão
resultado = serie_completa[
    (serie_completa['sale_date'] >= '2024-01-01') &
    (serie_completa['sale_date'] <= '2024-01-31')
][['sale_date', 'vendas', 'previsao']].reset_index(drop=True)

resultado.columns = ['data', 'vendas_reais', 'previsao']
resultado

,data,vendas_reais,previsao
0,2024-01-01,0.0,0.00
1,2024-01-02,0.0,0.00
2,2024-01-03,0.0,0.00
3,2024-01-04,0.0,0.00
4,2024-01-05,10.0,0.00
5,2024-01-06,0.0,1.43
6,2024-01-07,0.0,1.43
7,2024-01-08,0.0,1.43
8,2024-01-09,0.0,1.43
9,2024-01-10,0.0,1.43


In [5]:
# Concatena treino + teste para calcular a janela sem data leakage
serie_completa = pd.concat([treino, teste]).reset_index(drop=True)
serie_completa['previsao'] = (
    serie_completa['vendas']
    .shift(1)                        # shift garante que o dia atual não entra na janela
    .rolling(window=7, min_periods=1)
    .mean()
    .round(2)
)

# Extrai apenas o período de teste com a previsão
resultado = serie_completa[
    (serie_completa['sale_date'] >= '2024-01-01') &
    (serie_completa['sale_date'] <= '2024-01-31')
][['sale_date', 'vendas', 'previsao']].reset_index(drop=True)

resultado.columns = ['data', 'vendas_reais', 'previsao']
resultado

,data,vendas_reais,previsao
0,2024-01-01,0.0,0.00
1,2024-01-02,0.0,0.00
2,2024-01-03,0.0,0.00
3,2024-01-04,0.0,0.00
4,2024-01-05,10.0,0.00
5,2024-01-06,0.0,1.43
6,2024-01-07,0.0,1.43
7,2024-01-08,0.0,1.43
8,2024-01-09,0.0,1.43
9,2024-01-10,0.0,1.43


In [6]:
mae = np.mean(np.abs(resultado['vendas_reais'] - resultado['previsao']))
print(f"MAE — Janeiro 2024: {mae:.2f} unidades/dia")

MAE — Janeiro 2024: 0.65 unidades/dia


In [7]:
primeira_semana = resultado[resultado['data'] <= '2024-01-07']
soma_previsao = round(primeira_semana['previsao'].sum())
print(f"Soma da previsão (01/01 a 07/01/2024): {soma_previsao} unidades")

Soma da previsão (01/01 a 07/01/2024): 3 unidades



**METODOLOGIA — Questão 7**

1. COMO O BASELINE FOI CONSTRUÍDO?
   - Para cada dia do período de teste, a previsão é a média simples das
   vendas dos 7 dias imediatamente anteriores. Os dias sem venda registrada
   entram como zero no cálculo — o que reflete a realidade operacional
   de uma loja que abre todos os dias.

2. COMO EVITOU DATA LEAKAGE?
   - O .shift(1) desloca a série em 1 dia antes de aplicar o rolling(7),
   garantindo que o dia previsto nunca entra na janela de cálculo.
   O treino foi estritamente limitado a datas <= 31/12/2023, e nenhuma
   informação de Janeiro de 2024 foi usada para construir as previsões.

3. LIMITAÇÃO DO MODELO
   - A média móvel simples não captura sazonalidade nem tendência. Se o
   produto tem picos em determinados dias da semana ou meses do ano
   (como coletes no verão), o modelo ignora esse padrão e tende a
   subestimar os picos e superestimar os vales — exatamente o cenário
   que gerou o problema do estoque descrito no enunciado.